# Tomahawk PF/CS FEA + native-solver Kaggle run

This notebook is intentionally a two-file Kaggle workflow: upload this notebook and `Tomahawk_Kaggle_Data.zip` (or the file-library collision-suffixed copy of that ZIP). It uses the current PF/CS engineering closure, performs a real NumPy 3-D truss finite-element screen, and then attempts native-solver evidence only when a real solver package and MAKEGRID/mgrid file are present.

No Internet, `pip install`, synthetic equilibrium, or fabricated external-laboratory result is allowed. A missing solver is reported as an input condition. A completed FEA screen is not engineering qualification.

In [1]:
from pathlib import Path
import csv, hashlib, importlib, json, os, shutil, subprocess, sys, time, zipfile
import numpy as np

print('KAGGLE_BOOTSTRAP_START')
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd() / '_kaggle_work'
WORK_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working'), Path('/kaggle/models'), Path.cwd()]
SEARCH_ROOTS = [p for p in SEARCH_ROOTS if p.exists()]

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f'unsafe ZIP member: {member.filename}')
        z.extractall(destination)

def locate_data_root():
    archives = sorted({p.resolve() for root in SEARCH_ROOTS for p in root.rglob('Tomahawk_Kaggle_Data*.zip') if p.is_file()})
    if len(archives) > 1:
        raise RuntimeError('INPUT_BLOCKED: more than one Tomahawk_Kaggle_Data.zip found')
    if archives:
        extracted = WORK_DIR / 'tomahawk_kaggle_extracted'
        if extracted.exists():
            shutil.rmtree(extracted)
        extracted.mkdir(parents=True)
        safe_extract(archives[0], extracted)
        roots = sorted(p for p in extracted.rglob('kaggle_data_manifest.json') if p.is_file())
        if len(roots) != 1:
            raise RuntimeError('INPUT_BLOCKED: extracted data manifest is missing or ambiguous')
        return roots[0].parent
    roots = sorted({p.parent.resolve() for root in SEARCH_ROOTS for p in root.rglob('kaggle_data_manifest.json') if (p.parent / 'MHD/mhd_input.json').is_file()})
    if len(roots) != 1:
        raise RuntimeError('INPUT_BLOCKED: unpacked data root is missing or ambiguous')
    return roots[0]

try:
    DATA_ROOT = locate_data_root()
    data_manifest = json.loads((DATA_ROOT / 'kaggle_data_manifest.json').read_text())
    mismatches = []
    for record in data_manifest['files']:
        path = (DATA_ROOT / record['path']).resolve()
        if DATA_ROOT.resolve() not in path.parents or not path.is_file():
            mismatches.append(record['path'])
            continue
        if path.stat().st_size != record['size_bytes'] or sha256(path) != record['sha256']:
            mismatches.append(record['path'])
    required = ['MHD/mhd_input.json', 'MHD/mhd_profile_sweep.csv', 'MHD/cad_a2_coil_centerlines.csv', 'MHD/cad_a2_coil_currents.csv', 'CFD/cfd_case_manifest.json', 'CFD/cfd_cht_parameter_sweep.csv', 'CHF/chf_input.json', 'NEUTRONICS/material_region_map.csv', 'NEUTRONICS/openmc_input_manifest.json', 'NEUTRONICS/source_sweep.csv', 'MHD/engineering/cs_module_table.csv', 'MHD/engineering/cs_precompression_screen.json']
    missing = [name for name in required if not (DATA_ROOT / name).is_file()]
    if mismatches or missing:
        raise RuntimeError(json.dumps({'status': 'INPUT_BLOCKED', 'hash_mismatches': mismatches, 'missing': missing}))
    print('ONE_CLICK_INPUT_VERIFIED')
    print('data_root =', DATA_ROOT)
except Exception as exc:
    print(str(exc))
    raise

KAGGLE_BOOTSTRAP_START
ONE_CLICK_INPUT_VERIFIED
data_root = /kaggle/input/datasets/nicolebryant29/tomahawk-kaggle-data3/Tomahawk_Kaggle_Data


In [2]:
# Real NumPy 3-D truss FEA screen for the six-module CS stack.
# This is a transparent support/load model, not a released casing or coil FEA.

def read_csv_rows(path):
    with Path(path).open(newline='') as handle:
        return list(csv.DictReader(handle))

module_rows = read_csv_rows(DATA_ROOT / 'MHD/engineering/cs_module_table.csv')
preload_doc = json.loads((DATA_ROOT / 'MHD/engineering/cs_precompression_screen.json').read_text())
E = 150.0e9
REFERENCE_AREA_M2 = 0.10
ALLOWABLE_STRESS_PA = 250.0e6
DISPLACEMENT_LIMIT_M = 0.005
SUPPORT_OFFSET_M = 0.80
module_count = len(module_rows)
nodes = []
coil_nodes = []
anchor_nodes = []
for row in module_rows:
    i = int(row['module_index'])
    z = (i - 0.5 * (module_count - 1)) * float(row['module_height_m'])
    r = 0.5 * (float(row['inner_winding_radius_m']) + float(row['outer_winding_radius_m']))
    coil_nodes.append(len(nodes)); nodes.append([r, 0.0, z])
    anchors = []
    for theta, z_offset in zip((0.0, 0.5 * np.pi, np.pi, 1.5 * np.pi), (0.4, -0.2, 0.1, -0.3)):
        anchors.append(len(nodes))
        nodes.append([r + SUPPORT_OFFSET_M * np.cos(theta), SUPPORT_OFFSET_M * np.sin(theta), z + z_offset])
    anchor_nodes.extend(anchors)
nodes = np.asarray(nodes, dtype=float)
topology = [(coil, anchor, 'support') for i, coil in enumerate(coil_nodes) for anchor in anchor_nodes[4*i:4*i+4]]
topology += [(coil_nodes[i], coil_nodes[i+1], 'stack') for i in range(module_count - 1)]

def solve_case(module_forces, uniform_area_m2):
    elements = [(a, b, uniform_area_m2, kind) for a, b, kind in topology]
    K = np.zeros((3 * len(nodes), 3 * len(nodes)))
    for a, b, area, kind in elements:
        delta = nodes[b] - nodes[a]
        length = float(np.linalg.norm(delta))
        direction = delta / length
        block = E * area / length * np.outer(direction, direction)
        dofs_a = slice(3*a, 3*a+3); dofs_b = slice(3*b, 3*b+3)
        K[dofs_a, dofs_a] += block; K[dofs_b, dofs_b] += block
        K[dofs_a, dofs_b] -= block; K[dofs_b, dofs_a] -= block
    F = np.zeros(3 * len(nodes))
    for i, node in enumerate(coil_nodes):
        F[3*node + 2] = module_forces[i]
    fixed = sorted(dof for node in anchor_nodes for dof in (3*node, 3*node+1, 3*node+2))
    fixed_set = set(fixed)
    free = np.asarray([dof for dof in range(3 * len(nodes)) if dof not in fixed_set], dtype=int)
    A = K[np.ix_(free, free)]
    matrix_rank = int(np.linalg.matrix_rank(A))
    if matrix_rank != len(free):
        raise RuntimeError(f'FEA_INPUT_BLOCKED: singular support topology rank={matrix_rank}/{len(free)}')
    u = np.zeros(3 * len(nodes))
    u[free] = np.linalg.solve(A, F[free])
    element_results = []
    for a, b, area, kind in elements:
        delta = nodes[b] - nodes[a]; length = float(np.linalg.norm(delta)); direction = delta / length
        extension = float(np.dot(u[3*b:3*b+3] - u[3*a:3*a+3], direction))
        axial_force = E * area / length * extension
        element_results.append({'node_a': a, 'node_b': b, 'kind': kind, 'length_m': length, 'area_m2': area, 'axial_force_N': axial_force, 'stress_Pa': axial_force / area})
    max_stress = max(abs(item['stress_Pa']) for item in element_results)
    max_force = max(abs(item['axial_force_N']) for item in element_results)
    return {'max_displacement_m': float(np.max(np.abs(u))), 'max_element_stress_Pa': float(max_stress), 'max_element_stress_MPa': float(max_stress / 1.0e6), 'max_element_force_N': float(max_force), 'matrix_rank': matrix_rank, 'free_dof_count': int(len(free)), 'element_results': element_results}

def meets_screen(result):
    return result['max_element_stress_Pa'] <= ALLOWABLE_STRESS_PA and result['max_displacement_m'] <= DISPLACEMENT_LIMIT_M

case_results = {}
for scenario_name, scenario in preload_doc['scenarios'].items():
    forces = np.asarray(scenario['module_generalized_axial_force_N'], dtype=float)
    reference = solve_case(forces, REFERENCE_AREA_M2)
    low, high = 1.0e-4, REFERENCE_AREA_M2
    while high < 1.0e3 and not meets_screen(solve_case(forces, high)):
        high *= 2.0
    for _ in range(70):
        mid = 0.5 * (low + high)
        if meets_screen(solve_case(forces, mid)):
            high = mid
        else:
            low = mid
    sized = solve_case(forces, high)
    case_results[scenario_name] = {'reference_area_case': reference, 'sized_uniform_area_case': sized, 'required_uniform_area_m2': float(high), 'sized_case_meets_assumptions': bool(meets_screen(sized))}
worst = max(case_results, key=lambda name: case_results[name]['reference_area_case']['max_element_stress_Pa'])
fea_screen = {
    'schema_version': 'TOMAHAWK-KAGGLE-3D-TRUSS-FEA-SIZING-SCREEN-0.2',
    'status': 'CALCULATED_SUPPORT_SIZING_SCREEN_NOT_QUALIFIED',
    'model': {'element_type': '3-D axial truss support abstraction', 'nodes': int(len(nodes)), 'elements': int(len(topology)), 'anchors_per_module': 4, 'anchor_z_offsets_m': [0.4, -0.2, 0.1, -0.3], 'support_offset_m': SUPPORT_OFFSET_M, 'rank_check': 'full_rank_required_before_solve', 'E_Pa': E, 'reference_uniform_area_m2': REFERENCE_AREA_M2, 'allowable_stress_Pa_assumption': ALLOWABLE_STRESS_PA, 'displacement_limit_m_assumption': DISPLACEMENT_LIMIT_M, 'load_cases': list(preload_doc['scenarios'])},
    'results': {'worst_reference_case': worst, 'worst_reference_stress_MPa': case_results[worst]['reference_area_case']['max_element_stress_MPa'], 'worst_reference_displacement_m': case_results[worst]['reference_area_case']['max_displacement_m'], 'worst_required_uniform_area_m2': case_results[worst]['required_uniform_area_m2'], 'load_cases': case_results},
    'acceptance': {'linear_systems_solved': bool(all(item['reference_area_case']['matrix_rank'] == item['reference_area_case']['free_dof_count'] for item in case_results.values())), 'reference_geometry_pass': False, 'sized_math_case_meets_assumptions': bool(all(item['sized_case_meets_assumptions'] for item in case_results.values())), 'engineering_qualification': False},
    'limitations': ['This is a support-load sizing abstraction, not released casing or coil FEA.', 'Material properties and allowable limits are explicit screening assumptions.', 'No contact, preload, cooldown, Lorentz body-force map, casing, insulation, fatigue, fracture, mesh-convergence, or nonlinear behavior is solved.', 'The calculated area is a requirement for this abstraction, not a fabricated support design.']
}
print('SMOKE_PASS')
print('FEA support sizing:', fea_screen['results']['worst_reference_case'], '| reference stress MPa =', fea_screen['results']['worst_reference_stress_MPa'], '| required uniform area m2 =', fea_screen['results']['worst_required_uniform_area_m2'])

SMOKE_PASS
FEA support sizing: all_co_directional_conservative | reference stress MPa = 4812.593005560568 | required uniform area m2 = 1.925037202224227


In [3]:
# Native-solver adapter. It executes only an explicitly packaged, hashed solver.
solver_manifest = json.loads((DATA_ROOT / 'solver_package_manifest.json').read_text())
solver_type = solver_manifest.get('package_type', 'none')
native = {'schema_version': 'TOMAHAWK-KAGGLE-NATIVE-SOLVER-EVIDENCE-0.1', 'package_type': solver_type, 'status': None, 'mgrid': {}, 'execution': {}, 'qualification': {'free_boundary_3D_MHD_pass': False}}
mgrid_path = solver_manifest.get('mgrid_path')
if mgrid_path:
    mgrid = (DATA_ROOT / mgrid_path).resolve()
    if mgrid.is_file() and DATA_ROOT.resolve() in mgrid.parents:
        actual_mgrid_hash = sha256(mgrid)
        expected_mgrid_hash = solver_manifest.get('mgrid_sha256')
        native['mgrid'] = {'path': str(mgrid), 'sha256': actual_mgrid_hash, 'present': not expected_mgrid_hash or actual_mgrid_hash == expected_mgrid_hash, 'hash_match': not expected_mgrid_hash or actual_mgrid_hash == expected_mgrid_hash}
    else:
        native['mgrid'] = {'path': mgrid_path, 'present': False}
else:
    native['mgrid'] = {'present': False, 'reason': 'no mgrid_path declared'}

def run_hashed_binary():
    entry = (DATA_ROOT / solver_manifest['entrypoint']).resolve()
    if DATA_ROOT.resolve() not in entry.parents or not entry.is_file():
        return {'status': 'INPUT_BLOCKED', 'reason': 'binary entrypoint missing'}
    expected = solver_manifest.get('sha256')
    if not expected or sha256(entry) != expected:
        return {'status': 'INPUT_BLOCKED', 'reason': 'binary hash missing or mismatched'}
    args = solver_manifest.get('entrypoint_args', [])
    if not isinstance(args, list) or not all(isinstance(x, str) for x in args):
        return {'status': 'INPUT_BLOCKED', 'reason': 'entrypoint_args must be a string list'}
    try:
        result = subprocess.run([str(entry), *args], cwd=str(entry.parent), capture_output=True, text=True, timeout=3600)
    except Exception as exc:
        return {'status': 'EXECUTION_ERROR', 'reason': str(exc)}
    (WORK_DIR / 'native_solver_stdout.txt').write_text(result.stdout)
    (WORK_DIR / 'native_solver_stderr.txt').write_text(result.stderr)
    return {'status': 'NATIVE_SOLVER_EXECUTED' if result.returncode == 0 else 'NATIVE_SOLVER_FAILED', 'returncode': result.returncode, 'stdout_path': str(WORK_DIR / 'native_solver_stdout.txt'), 'stderr_path': str(WORK_DIR / 'native_solver_stderr.txt')}

if solver_type == 'binary':
    native['execution'] = run_hashed_binary()
elif solver_type == 'python_module':
    module_root = (DATA_ROOT / solver_manifest.get('module_root', '')).resolve()
    if module_root.is_dir() and DATA_ROOT.resolve() in module_root.parents:
        sys.path.insert(0, str(module_root))
        try:
            importlib.import_module(solver_manifest['module'])
            native['execution'] = {'status': 'MODULE_IMPORT_PASS', 'module': solver_manifest['module'], 'note': 'A solver callable and declared outputs are still required for execution evidence.'}
        except Exception as exc:
            native['execution'] = {'status': 'NATIVE_SOLVER_FAILED', 'reason': repr(exc)}
    else:
        native['execution'] = {'status': 'INPUT_BLOCKED', 'reason': 'packaged module_root missing'}
elif solver_type == 'source_archive':
    native['execution'] = {'status': 'INPUT_BLOCKED_SOURCE_BUILD_REQUIRED', 'reason': 'source archives are not compiled or installed automatically; provide a reproducible local build contract and generated executable'}
else:
    native['execution'] = {'status': 'INPUT_BLOCKED', 'reason': 'no native solver package declared'}
native['status'] = native['execution']['status']
native['qualification']['mgrid_present'] = bool(native['mgrid'].get('present'))
native['qualification']['native_solver_executed'] = native['status'] == 'NATIVE_SOLVER_EXECUTED'
native['qualification']['free_boundary_3D_MHD_pass'] = bool(native['qualification']['mgrid_present'] and native['qualification']['native_solver_executed'])
print('NATIVE_SOLVER_STATUS:', native['status'])
print('native solver pass:', native['qualification']['free_boundary_3D_MHD_pass'])

NATIVE_SOLVER_STATUS: INPUT_BLOCKED
native solver pass: False


In [4]:
# Write a self-contained evidence package for Claude or another reviewer.
results_dir = WORK_DIR / 'Tomahawk_Kaggle_Results'
if results_dir.exists():
    shutil.rmtree(results_dir)
results_dir.mkdir(parents=True)
def write_json(name, doc):
    (results_dir / name).write_text(json.dumps(doc, indent=2) + '\n')
write_json('fea_screen_results.json', fea_screen)
write_json('native_solver_evidence.json', native)
write_json('run_summary.json', {
    'schema_version': 'TOMAHAWK-KAGGLE-FEA-NATIVE-RUN-0.1',
    'status': 'FEA_SCREEN_COMPLETE_NATIVE_SOLVER_OPEN' if not native['qualification']['free_boundary_3D_MHD_pass'] else 'FEA_AND_NATIVE_SOLVER_EVIDENCE_COMPLETE_QUALIFICATION_OPEN',
    'data_root': str(DATA_ROOT),
    'fea_reference_geometry_pass': fea_screen['acceptance']['reference_geometry_pass'],
    'fea_sized_math_case_meets_assumptions': fea_screen['acceptance']['sized_math_case_meets_assumptions'],
    'native_solver_status': native['status'],
    'free_boundary_3D_MHD_pass': False,
    'qualified_design_pass': False,
    'external_execution_claim': False,
    'next_required_inputs': data_manifest['required_not_synthesized']
})
with zipfile.ZipFile(WORK_DIR / 'Tomahawk_Kaggle_Results.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(results_dir.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(results_dir).as_posix())
print('FULL_RUN_COMPLETE')
print('FEA_SCREEN_EXECUTION_PASS' if fea_screen['acceptance']['linear_systems_solved'] else 'SCREENING_FAIL_REVISE')
print('SCREENING_COMPLETE_WITH_EXTERNAL_BLOCKERS' if fea_screen['acceptance']['linear_systems_solved'] and not native['qualification']['free_boundary_3D_MHD_pass'] else 'NATIVE_SOLVER_EVIDENCE_COMPLETE_OR_FEA_REVISE')
print('RESULT_ZIP:', WORK_DIR / 'Tomahawk_Kaggle_Results.zip')

FULL_RUN_COMPLETE
FEA_SCREEN_EXECUTION_PASS
SCREENING_COMPLETE_WITH_EXTERNAL_BLOCKERS
RESULT_ZIP: /kaggle/working/Tomahawk_Kaggle_Results.zip
